In [ ]:
import requests, os

REPO = "https://raw.githubusercontent.com/ASM205/sector-intelligence/main/core"
FILES = ["config.py", "ingest.py", "pipeline.py"]

for f in FILES:
    r = requests.get(f"{REPO}/{f}")
    with open(f"/lakehouse/default/Files/{f}", "w") as out:
        out.write(r.text)
    print(f"✓ Downloaded {f}")

In [ ]:
import getpass
os.environ["ALPACA_KEY"]    = getpass.getpass("Enter Alpaca API key: ")
os.environ["ALPACA_SECRET"] = getpass.getpass("Enter Alpaca secret:  ")
print("✓ Keys set")

In [ ]:
from notebookutils import mssparkutils

mssparkutils.fs.mkdirs("abfss://default@onelake.dfs.fabric.microsoft.com/bronze")
mssparkutils.fs.mkdirs("abfss://default@onelake.dfs.fabric.microsoft.com/silver")
mssparkutils.fs.mkdirs("abfss://default@onelake.dfs.fabric.microsoft.com/gold")
print("✓ Lakehouse folders created")

In [ ]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS bronze_bars_v2 (
        ticker       STRING,
        sector       STRING,
        sub_industry STRING,
        granularity  STRING,
        t            BIGINT,
        o            DOUBLE,
        h            DOUBLE,
        l            DOUBLE,
        c            DOUBLE,
        v            DOUBLE,
        vw           DOUBLE,
        n            BIGINT
    )
    USING DELTA
""")
print("✓ Bronze table created")

In [ ]:
for table in ["Avi.dbo.silver_daily", "Avi.dbo.silver_5min", "Avi.dbo.silver_1min"]:
    spark.sql(f"CREATE TABLE IF NOT EXISTS {table} USING DELTA AS SELECT * FROM Avi.dbo.bronze_bars_v2 WHERE 1=0")
    print(f"✓ {table} created")

In [ ]:
for table in [
    "Avi.dbo.gold_sector_daily",
    "Avi.dbo.gold_ticker_signals",
    "Avi.dbo.gold_rankings_daily",
    "Avi.dbo.gold_correlation_daily",
    "Avi.dbo.gold_intraday_summary",
    "Avi.dbo.dashboard_measures",
]:
    spark.sql(f"CREATE TABLE IF NOT EXISTS {table} USING DELTA AS SELECT * FROM Avi.dbo.bronze_bars_v2 WHERE 1=0")
    print(f"✓ {table} created")

In [ ]:
import sys
sys.path.insert(0, "/lakehouse/default/Files")

from pipeline import run_pipeline
run_pipeline()